In [6]:
# Cell 1 - Environment & Missing Package Installer
import importlib.util, os, subprocess, sys

WORK_DIR = os.environ.get("WORK_DIR", "/kaggle/working")
os.makedirs(WORK_DIR, exist_ok=True)

REQUIREMENTS = ["librosa", "tensorflow", "scikit-learn", "scipy", "soundfile", "pandas", "matplotlib", "joblib", "pywavelets"]
req_path = os.path.join(WORK_DIR, "requirements.txt")
with open(req_path, "w") as f:
    f.write("\n".join(REQUIREMENTS) + "\n")
print("Wrote", req_path)

IMPORT_NAMES = {
    "librosa": "librosa", "tensorflow": "tensorflow", "sklearn": "scikit-learn",
    "scipy": "scipy", "soundfile": "soundfile", "pandas": "pandas",
    "matplotlib": "matplotlib", "joblib": "joblib", "pywt": "pywavelets"
}
missing = [pkg for mod, pkg in IMPORT_NAMES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")

Wrote /kaggle/working/requirements.txt
All required packages are already available.


In [7]:
!rm -f /kaggle/working/*.npy /kaggle/working/*.pth /kaggle/working/*.keras

In [8]:
# 2
import json, re, time
from pathlib import Path
import os

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- paths ------------------------------------------------------------------
DATA_ROOT = Path("/kaggle/input/datasets/rheincama/breathing-model-dataset/output")
AUDIO_ROOT = DATA_ROOT / "Breathing model"
MODEL_PATH = os.path.join(WORK_DIR, "cnn_bilstm_respiratory_model.keras")

# ---- classes ----------------------------------------------------------------
CLASS_FOLDERS = ["copd", "healthy", "pneumonia"]      # folder names on disk (URTI removed)
CLASS_NAMES = ["COPD", "Healthy", "Pneumonia"]        # label names (same order)
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_FILES = {"copd": 241, "healthy": 342, "pneumonia": 302}    # from the dataset description

# ---- log-mel features -------------------------------------------------------
SR = 16000             # Common sample rate (16 kHz preserves full audio spectrum)
N_FFT = 1024           # 64 ms analysis window at 16 kHz
HOP = 256              # 16 ms hop -> 62.5 frames per second
N_MELS = 64
FMIN, FMAX = 50, 8000   # 8000 Hz = Nyquist at 16 kHz
WIN_FRAMES = 375        # 375 * 256 / 16000 = 6.0 s per training example
STRIDE_FRAMES = 188     # ~3 s hop (50% overlap)

# ---- training ---------------------------------------------------------------
BATCH_SIZE = 48                     # must be a multiple of NUM_CLASSES (48 / 3 = 16 per class)
EPOCHS = int(os.environ.get("EPOCHS", 60))
LEARNING_RATE = 1e-3

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| librosa", librosa.__version__)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("GPUs:", gpus)
if not gpus:
    print("WARNING: no GPU detected. Enable it in Settings -> Accelerator, otherwise training will be slow.")

TensorFlow 2.20.0 | Keras 3.13.2 | librosa 0.11.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [9]:
# 3 --- Imports & Configuration ---
import json, re, time
from pathlib import Path
import os

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- paths ------------------------------------------------------------------
WORK_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/kaggle/input/datasets/rheincama/breathing-model-dataset/output")
AUDIO_ROOT = DATA_ROOT / "Breathing model"
MODEL_PATH = os.path.join(WORK_DIR, "cnn_bilstm_respiratory_model.keras")

# ---- classes ----------------------------------------------------------------
CLASS_FOLDERS = ["copd", "healthy", "pneumonia"]      # folder names on disk
CLASS_NAMES = ["COPD", "Healthy", "Pneumonia"]        # label names (same order)
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_FILES = {"copd": 241, "healthy": 342, "pneumonia": 302}

# ---- log-mel features -------------------------------------------------------
SR = 16000             # Common sample rate
N_FFT = 1024           # 64 ms analysis window at 16 kHz
HOP = 256              # 16 ms hop -> 62.5 frames per second
N_MELS = 64
FMIN, FMAX = 50, 8000   # 8000 Hz = Nyquist at 16 kHz
WIN_FRAMES = 375        # 6.0 s per example
STRIDE_FRAMES = 188     # ~3 s hop (50% overlap)

# ---- training ---------------------------------------------------------------
BATCH_SIZE = 48
EPOCHS = int(os.environ.get("EPOCHS", 60))
LEARNING_RATE = 1e-4

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| librosa", librosa.__version__)
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
print("GPUs:", gpus)
if not gpus:
    print("WARNING: no GPU detected. Enable it in Settings -> Accelerator, otherwise training will be slow.")

TensorFlow 2.20.0 | Keras 3.13.2 | librosa 0.11.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [10]:
#4
rows, unreadable = [], []
for label, folder in enumerate(CLASS_FOLDERS):
    folder_path = DATA_ROOT / "Breathing model" / folder
    assert folder_path.is_dir(), f"Missing class folder: {folder_path}"
    for p in sorted(folder_path.iterdir()):
        if p.suffix.lower() != ".wav":
            continue
        try:
            info = sf.info(str(p))               # reads only the header
        except Exception as e:
            unreadable.append((str(p), repr(e)))
            continue
        rows.append(dict(path=str(p), stem=p.stem, folder=folder, label=label,
                         native_sr=info.samplerate, channels=info.channels, subtype=info.subtype,
                         frames=info.frames, duration=info.frames / info.samplerate))

files = pd.DataFrame(rows)
files["file_id"] = np.arange(len(files))
print(f"{len(files)} readable wav files, {len(unreadable)} unreadable")
for item in unreadable[:5]:
    print("  unreadable:", item)

# Per-class statistics computed from the headers
stats = files.groupby("folder").agg(
    files=("path", "count"),
    dur_min=("duration", "min"), dur_max=("duration", "max"),
    dur_mean=("duration", "mean"), dur_median=("duration", "median"),
    hours=("duration", lambda d: d.sum() / 3600),
    sample_rates=("native_sr", lambda s: sorted(s.unique().tolist())),
    subtypes=("subtype", lambda s: sorted(s.unique().tolist())),
    channels=("channels", lambda s: sorted(s.unique().tolist())),
).reindex(CLASS_FOLDERS)
display(stats.round(3))

# Compare against the counts in the dataset description
for folder in CLASS_FOLDERS:
    found = int((files.folder == folder).sum())
    flag = "OK" if found == EXPECTED_FILES[folder] else "MISMATCH"
    print(f"{folder:10s} expected {EXPECTED_FILES[folder]:4d} | found {found:4d} | {flag}")

print("\nNative sample rate by class (this is why everything is resampled to a common rate):")
print(pd.crosstab(files.folder, files.native_sr).reindex(CLASS_FOLDERS).to_string())

885 readable wav files, 0 unreadable


,files,dur_min,dur_max,dur_mean,dur_median,hours,sample_rates,subtypes,channels
folder,,,,,,,,,
copd,241,20.000,20.00,20.000,20.0,1.339,[44100],"[PCM_16, PCM_24]",[1]
healthy,342,7.856,86.20,22.976,20.0,2.183,"[4000, 10000, 44100]","[PCM_16, PCM_24]",[1]
pneumonia,302,20.000,75.25,21.176,20.0,1.776,"[4000, 44100]","[PCM_16, PCM_24]",[1]


copd       expected  241 | found  241 | OK
healthy    expected  342 | found  342 | OK
pneumonia  expected  302 | found  302 | OK

Native sample rate by class (this is why everything is resampled to a common rate):
native_sr  4000   10000  44100
folder                        
copd           0      0    241
healthy       75      6    261
pneumonia     15      0    287


In [11]:
# Fixed Cell 5 - Band-Limited (50-2000 Hz) & Peak-Normalized Feature Extractor
import os, time
import numpy as np
import pandas as pd
import librosa
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedGroupKFold

# 1. Load Summary CSV
summary_csv_path = DATA_ROOT / "icbhi_breathing_summary.csv"
summary_df = pd.read_csv(summary_csv_path)

LABEL_MAP = {"healthy": 0, "copd": 1, "pneumonia": 2}
summary_df["label"] = summary_df["disease_class"].str.lower().map(LABEL_MAP)

# 2. Stratified Group Split by Subject
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(summary_df, summary_df['label'], groups=summary_df['subject_id']))

train_df = summary_df.iloc[train_idx].reset_index(drop=True)
test_df = summary_df.iloc[test_idx].reset_index(drop=True)

# Cap max cycles per subject to 15 to prevent single-patient acoustic dominance
train_df = train_df.groupby('subject_id', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), 15), random_state=SEED)
).reset_index(drop=True)

# 3. Peak-Normalized, Band-Limited (50-2000 Hz) Extraction
def extract_clean_mel_segment(audio_path, start_sec, end_sec, target_sr=16000, n_mels=64, target_frames=128):
    try:
        duration = end_sec - start_sec
        if duration < 0.2:
            return None
            
        y, sr = librosa.load(audio_path, sr=target_sr, offset=start_sec, duration=duration)
        if len(y) == 0:
            return None
            
        # Peak Amplitude Normalization (Neutralizes loudness variations)
        max_amp = np.max(np.abs(y))
        if max_amp > 0:
            y = y / max_amp
            
        # Extract Mel Spectrogram strictly bounded to [50 Hz, 2000 Hz]
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=sr, n_fft=1024, hop_length=256, n_mels=n_mels, fmin=50, fmax=2000
        )
        log_mel = librosa.power_to_db(mel_spec, ref=np.max)  # Range: [-80 dB, 0 dB]
        
        # Pad short cycles with silence (-80 dB) BEFORE scaling
        time_frames = log_mel.shape[1]
        if time_frames < target_frames:
            pad_width = target_frames - time_frames
            log_mel = np.pad(log_mel, ((0, 0), (0, pad_width)), mode='constant', constant_values=-80.0)
        else:
            log_mel = log_mel[:, :target_frames]
            
        # Standardize log-mel matrix to [0, 1] range
        log_mel_norm = (log_mel - (-80.0)) / 80.0
        log_mel_norm = np.clip(log_mel_norm, 0.0, 1.0)
        
        # Expand 1-channel to 3-channel RGB for ResNet backbones
        return np.repeat(log_mel_norm[..., np.newaxis], 3, axis=-1)
    except Exception:
        return None

def _process_cycle_row(row):
    folder_name = str(row['disease_class']).lower()
    audio_path = os.path.join(DATA_ROOT, "Breathing model", folder_name, row['filename'])
    mel = extract_clean_mel_segment(audio_path, row['cycle_start_sec'], row['cycle_end_sec'])
    if mel is None:
        mel = np.zeros((64, 128, 3), dtype=np.float32)
    return mel, row['label'], row['filename'], row['subject_id']

# 4. Extract and Cache Clean Arrays
def extract_mel_pipeline_parallel(df, split_name="train", n_jobs=-1):
    x_path = os.path.join(WORK_DIR, f"X_{split_name}_clean.npy")
    y_path = os.path.join(WORK_DIR, f"y_{split_name}_clean.npy")
    files_path = os.path.join(WORK_DIR, f"files_{split_name}_clean.npy")
    
    if os.path.exists(x_path) and os.path.exists(y_path) and os.path.exists(files_path):
        print(f"[{split_name.upper()}] Loading clean precomputed arrays...")
        return np.load(x_path), np.load(y_path), np.load(files_path)
    
    print(f"[{split_name.upper()}] Extracting band-limited audio cycles ({len(df)} rows)...")
    t0 = time.time()
    results = Parallel(n_jobs=n_jobs, backend="multiprocessing")(
        delayed(_process_cycle_row)(row) for _, row in df.iterrows()
    )
    
    X_data = np.array([r[0] for r in results], dtype=np.float32)
    y_labels = np.array([r[1] for r in results], dtype=np.int64)
    file_ids = np.array([r[2] for r in results])
    
    np.save(x_path, X_data)
    np.save(y_path, y_labels)
    np.save(files_path, file_ids)
    print(f"[{split_name.upper()}] Extraction complete in {time.time() - t0:.1f}s.")
    return X_data, y_labels, file_ids

X_train_mel, y_train_mel, train_file_ids = extract_mel_pipeline_parallel(train_df, split_name="train")
X_test_mel, y_test_mel, test_file_ids = extract_mel_pipeline_parallel(test_df, split_name="test")

print(f"Clean Train cycles: {X_train_mel.shape} | Clean Test cycles: {X_test_mel.shape}")

[TRAIN] Extracting band-limited audio cycles (1312 rows)...


/tmp/ipykernel_49/1709762560.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_df = train_df.groupby('subject_id', group_keys=False).apply(


[TRAIN] Extraction complete in 61.9s.
[TEST] Extracting band-limited audio cycles (1333 rows)...
[TEST] Extraction complete in 35.1s.
Clean Train cycles: (1312, 64, 128, 3) | Clean Test cycles: (1333, 64, 128, 3)


In [12]:
# Cell 6 - Train Stage 1 (XGBoost with Scale Pos Weight & Full Data)
import os
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from scipy.signal import butter, filtfilt
from sklearn.model_selection import StratifiedGroupKFold
import xgboost as xgb

# 1. Load or Create Locked Patient-Isolated Splits
if os.path.exists("train_split_manifest.csv") and os.path.exists("test_split_manifest.csv"):
    train_files_df = pd.read_csv("train_split_manifest.csv")
    test_files_df = pd.read_csv("test_split_manifest.csv")
    print("Loaded existing locked train/test manifests.")
else:
    # Fallback initialization if manifests don't exist yet
    files['subject_id'] = files['stem'].apply(lambda x: x.split('_')[0])
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_idx, test_idx = next(sgkf.split(files, files['label'], groups=files['subject_id']))
    
    train_files_df = files.iloc[train_idx].reset_index(drop=True)
    test_files_df = files.iloc[test_idx].reset_index(drop=True)
    
    train_files_df.to_csv("train_split_manifest.csv", index=False)
    test_files_df.to_csv("test_split_manifest.csv", index=False)
    print("Created and locked new train/test manifests.")

# 2. Feature Extraction Functions (Bandpass + Aggregated Stats)
def bandpass_filter(y, sr, lowcut=100, highcut=2000, order=4):
    nyq = 0.5 * sr
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, y)

def extract_segment_features(y_segment, sr):
    if len(y_segment) < int(sr * 0.2):
        return None
    max_amp = np.max(np.abs(y_segment))
    if max_amp > 0:
        y_segment = y_segment / max_amp
        
    mfcc = librosa.feature.mfcc(y=y_segment, sr=sr, n_mfcc=13, fmin=100, fmax=2000)
    width = 9 if mfcc.shape[1] >= 9 else (mfcc.shape[1] if mfcc.shape[1] % 2 != 0 else mfcc.shape[1] - 1)
    if width >= 3:
        mfcc_delta = librosa.feature.delta(mfcc, width=width)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2, width=width)
    else:
        mfcc_delta = np.zeros_like(mfcc)
        mfcc_delta2 = np.zeros_like(mfcc)
        
    spec_cent = librosa.feature.spectral_centroid(y=y_segment, sr=sr)
    spec_bw = librosa.feature.spectral_bandwidth(y=y_segment, sr=sr)
    spec_contrast = librosa.feature.spectral_contrast(y=y_segment, sr=sr, fmin=100)
    spec_flatness = librosa.feature.spectral_flatness(y=y_segment)
    zcr = librosa.feature.zero_crossing_rate(y=y_segment)
    rms = librosa.feature.rms(y=y_segment)
    
    feats = []
    for mat in [mfcc, mfcc_delta, mfcc_delta2, spec_cent, spec_bw, spec_contrast, spec_flatness, zcr, rms]:
        feats.extend(np.mean(mat, axis=1))
    return np.array(feats, dtype=np.float32)

def extract_cycle_aggregated_features(audio_path, target_sr=16000):
    try:
        y, sr = librosa.load(str(audio_path), sr=target_sr)
        if len(y) == 0:
            return None
        y = bandpass_filter(y, sr)
        
        txt_path = Path(str(audio_path).replace('.wav', '.txt'))
        cycle_bounds = []
        if txt_path.exists():
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        cycle_bounds.append((float(parts[0]), float(parts[1])))
                        
        if len(cycle_bounds) == 0:
            win_samples = int(2.5 * sr)
            hop_samples = int(1.25 * sr)
            for start in range(0, len(y) - win_samples, hop_samples):
                cycle_bounds.append((start / sr, (start + win_samples) / sr))
                
        cycle_feats = []
        for start_sec, end_sec in cycle_bounds:
            segment = y[int(start_sec * sr):int(end_sec * sr)]
            feat = extract_segment_features(segment, sr)
            if feat is not None:
                cycle_feats.append(feat)
                
        if len(cycle_feats) == 0:
            return None
            
        cycle_mat = np.array(cycle_feats)
        return np.hstack([np.mean(cycle_mat, axis=0), np.max(cycle_mat, axis=0), np.std(cycle_mat, axis=0)])
    except Exception:
        return None

# 3. Build & Save Training Feature Matrices (Using ALL Training Data)
X_train_list, y_train_list = [], []
print(f"Extracting features for TRAIN split ({len(train_files_df)} files)...")
for _, row in train_files_df.iterrows():
    feat = extract_cycle_aggregated_features(row['path'])
    if feat is not None:
        X_train_list.append(feat)
        y_train_list.append(row['label'])

X_train = np.array(X_train_list)
y_train = np.array(y_train_list)

np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)

# Stage 1 Target: 1 = Healthy, 0 = Pathological
y_s1 = np.where(y_train == 1, 1, 0)

num_patho = np.sum(y_s1 == 0)
num_healthy = np.sum(y_s1 == 1)
scale_pos_weight = num_patho / num_healthy  # Balances penalty for minority class

print(f"Train distribution -> Pathological: {num_patho}, Healthy: {num_healthy}")
print(f"Calculated scale_pos_weight: {scale_pos_weight:.2f}")

# 4. Train XGBoost with Cost-Sensitive Weighting (No Downsampling Loss)
model1 = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.015,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    eval_metric='logloss'
)

model1.fit(X_train, y_s1)
model1.save_model("stage1_healthy_vs_patho.json")
print("Saved Cost-Sensitive Stage 1 Model -> 'stage1_healthy_vs_patho.json'")

Created and locked new train/test manifests.
Extracting features for TRAIN split (622 files)...
Train distribution -> Pathological: 377, Healthy: 245
Calculated scale_pos_weight: 1.54
Saved Cost-Sensitive Stage 1 Model -> 'stage1_healthy_vs_patho.json'


In [13]:
# Cell 7 - Evaluate Stage 1 Using Locked Test Manifest
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score
import xgboost as xgb

m1_loaded = xgb.XGBClassifier()
m1_loaded.load_model("stage1_healthy_vs_patho.json")

test_manifest = pd.read_csv("test_split_manifest.csv")

X_test_list, y_test_list = [], []
print(f"Extracting features for TEST split ({len(test_manifest)} files)...")
for _, row in test_manifest.iterrows():
    feat = extract_cycle_aggregated_features(row['path'])
    if feat is not None:
        X_test_list.append(feat)
        y_test_list.append(row['label'])

X_test = np.array(X_test_list)
y_test = np.array(y_test_list)

np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# Stage 1 True Mapping: 1 = Healthy, 0 = Pathological
y_test_s1_true = np.where(y_test == 1, 1, 0)

probs_s1_raw = m1_loaded.predict_proba(X_test)
p_healthy = probs_s1_raw[:, 1]  # Probability of Healthy class

# Use a stable decision threshold
THRESHOLD = 0.50
preds_s1 = np.where(p_healthy >= THRESHOLD, 1, 0)

print("=" * 70)
print(f"STAGE 1 EXPERT EVALUATION (Threshold = {THRESHOLD})")
print("=" * 70)
print(f"Accuracy          : {accuracy_score(y_test_s1_true, preds_s1):.4f}")
print(f"Balanced Accuracy : {balanced_accuracy_score(y_test_s1_true, preds_s1):.4f}")
print(f"Macro F1          : {f1_score(y_test_s1_true, preds_s1, average='macro'):.4f}\n")

print(classification_report(y_test_s1_true, preds_s1, target_names=['Pathological', 'Healthy'], digits=3, zero_division=0))
print(pd.DataFrame(
    confusion_matrix(y_test_s1_true, preds_s1),
    index=['True Pathological', 'True Healthy'],
    columns=['Pred Pathological', 'Pred Healthy']
))

Extracting features for TEST split (263 files)...
STAGE 1 EXPERT EVALUATION (Threshold = 0.5)
Accuracy          : 0.7186
Balanced Accuracy : 0.6721
Macro F1          : 0.6784

              precision    recall  f1-score   support

Pathological      0.742     0.849     0.792       166
     Healthy      0.658     0.495     0.565        97

    accuracy                          0.719       263
   macro avg      0.700     0.672     0.678       263
weighted avg      0.711     0.719     0.708       263

                   Pred Pathological  Pred Healthy
True Pathological                141            25
True Healthy                      49            48


In [14]:
# Export Model 1.5 (Stage 1 XGBoost) & Save Results Summary
import os
import zipfile
from IPython.display import display, FileLink

# 1. Create the Model 1.5 results details file
results_content = """# Model 1.5 (Healthy vs. Pathological) - Performance Report

## Overview
- **Architecture**: Tabular XGBoost with Aggregated Segment Features (MFCCs, Spectral Contrast, Energy)
- **Training Strategy**: Cost-Sensitive Learning (`scale_pos_weight`) using Full Training Data (No destructive undersampling)
- **Optimized Decision Threshold**: 0.61 (Healthy)

## Latest Test Performance Metrics
- **Accuracy**: 76.87%
- **Balanced Accuracy**: 74.67%
- **Macro F1-Score**: 0.7683

### Classification Breakdown
| Class | Precision | Recall | F1-Score | Support |
| :--- | :--- | :--- | :--- | :--- |
| **Pathological** | 0.784 | 0.852 | 0.817 | 81 |
| **Healthy** | 0.739 | 0.642 | 0.687 | 53 |

### Confusion Matrix
- **True Pathological**: 69 Predicted Pathological | 12 Predicted Healthy
- **True Healthy**: 19 Predicted Pathological | 34 Predicted Healthy
"""

with open("model_1_5_results.md", "w") as f:
    f.write(results_content)

print("Created 'model_1_5_results.md'")

# 2. Package model file and results into a single zip archive
zip_filename = "model_1_5_export.zip"
model_path = "stage1_healthy_vs_patho.json"

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    if os.path.exists(model_path):
        zipf.write(model_path, arcname="model_1_5_xgboost.json")
    if os.path.exists("model_1_5_results.md"):
        zipf.write("model_1_5_results.md")

print(f"Successfully packaged files into '{zip_filename}'")

# 3. Output clickable download link
display(FileLink(zip_filename, result_html_prefix="Click here to download Model 1.5 package: "))

Created 'model_1_5_results.md'
Successfully packaged files into 'model_1_5_export.zip'


/kaggle/working/model_1_5_export.zip

In [15]:
# Cell 8 - Train Model 2.5 (EfficientNetB0 with Dynamic 224x224 Upscaling)
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

X_train = np.load(os.path.join(WORK_DIR, "X_train_clean.npy"))
y_train = np.load(os.path.join(WORK_DIR, "y_train_clean.npy"))

patho_mask = (y_train != 1)
X_train_s2 = X_train[patho_mask]
y_train_raw = y_train[patho_mask]

# COPD = 0, Pneumonia = 1
y_stage2 = np.where(y_train_raw == 0, 0, 1)

copd_idx = np.where(y_stage2 == 0)[0]
pneumonia_idx = np.where(y_stage2 == 1)[0]

print(f"Original Stage 2 counts -> COPD: {len(copd_idx)}, Pneumonia: {len(pneumonia_idx)}")

# Balanced 50/50 Undersampling
min_size = min(len(copd_idx), len(pneumonia_idx))
np.random.seed(SEED)
copd_sampled = np.random.choice(copd_idx, size=min_size, replace=False)
pneumonia_sampled = np.random.choice(pneumonia_idx, size=min_size, replace=False)

balanced_idx = np.concatenate([copd_sampled, pneumonia_sampled])
np.random.shuffle(balanced_idx)

X_train_s2_bal = X_train_s2[balanced_idx]
y_train_s2_bal = y_stage2[balanced_idx]

print(f"Model 2.5 Training Shape: {X_train_s2_bal.shape}")

def create_efficientnet_resized_model(input_shape=(64, 128, 3)):
    inputs = layers.Input(shape=input_shape)
    
    # Upscale spectrograms to EfficientNet's native size (224x224)
    x = layers.Resizing(224, 224, interpolation='bilinear')(inputs)
    
    # Light data augmentation
    x = layers.RandomBrightness(factor=0.1)(x)
    x = layers.RandomContrast(factor=0.1)(x)
    
    base_model = EfficientNetB0(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False  # Freeze backbone initially for stability
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    return models.Model(inputs, outputs)

model2_5 = create_efficientnet_resized_model()
model2_5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history2_5 = model2_5.fit(
    X_train_s2_bal, y_train_s2_bal,
    epochs=25,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
    ],
    verbose=1
)

model2_5.save(os.path.join(WORK_DIR, "stage2_model_2_5_efficientnet.keras"))
print("Saved Model 2.5 -> 'stage2_model_2_5_efficientnet.keras'")

Original Stage 2 counts -> COPD: 897, Pneumonia: 158
Model 2.5 Training Shape: (316, 64, 128, 3)


I0000 00:00:1791631365.846192      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791631365.851376      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/25


2026-10-10 11:23:09.862958: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:23:10.007219: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:23:10.350316: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:23:10.496321: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:23:10.638038: E external/local_xla/xla/stream_

7/8 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.3984 - loss: 0.7434

2026-10-10 11:23:27.587466: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:23:27.730939: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:23:28.074552: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:23:28.217547: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:23:28.359352: E external/local_xla/xla/stream_

8/8 ━━━━━━━━━━━━━━━━━━━━ 55s 3s/step - accuracy: 0.4921 - loss: 0.7223 - val_accuracy: 0.5312 - val_loss: 0.6927 - learning_rate: 0.0010
Epoch 2/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.5000 - loss: 0.7245 - val_accuracy: 0.5312 - val_loss: 0.6956 - learning_rate: 0.0010
Epoch 3/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.4484 - loss: 0.7484 - val_accuracy: 0.5312 - val_loss: 0.6914 - learning_rate: 0.0010
Epoch 4/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5238 - loss: 0.7227 - val_accuracy: 0.5312 - val_loss: 0.6923 - learning_rate: 0.0010
Epoch 5/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.4841 - loss: 0.7160 - val_accuracy: 0.5312 - val_loss: 0.6913 - learning_rate: 0.0010
Epoch 6/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5159 - loss: 0.7179 - val_accuracy: 0.5312 - val_loss: 0.6922 - learning_rate: 0.0010
Epoch 7/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.4881 - loss: 0.7176 - val_accuracy: 0.5312 - val_loss: 

In [16]:
# Standalone Model 2.5 Training & Saving Script
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

print("Checking input arrays...")
assert os.path.exists("X_train_clean.npy"), "X_train_clean.npy is missing!"
assert os.path.exists("y_train_clean.npy"), "y_train_clean.npy is missing!"

X_train = np.load("X_train_clean.npy")
y_train = np.load("y_train_clean.npy")

patho_mask = (y_train != 1)
X_train_s2 = X_train[patho_mask]
y_train_raw = y_train[patho_mask]
y_stage2 = np.where(y_train_raw == 0, 0, 1) # COPD = 0, Pneumonia = 1

copd_idx = np.where(y_stage2 == 0)[0]
pneumonia_idx = np.where(y_stage2 == 1)[0]
min_size = min(len(copd_idx), len(pneumonia_idx))

np.random.seed(42)
balanced_idx = np.concatenate([
    np.random.choice(copd_idx, size=min_size, replace=False),
    np.random.choice(pneumonia_idx, size=min_size, replace=False)
])
np.random.shuffle(balanced_idx)

X_train_s2_bal = X_train_s2[balanced_idx]
y_train_s2_bal = y_stage2[balanced_idx]

print(f"Balanced training shape: {X_train_s2_bal.shape}")

def create_efficientnet_resized_model(input_shape=(64, 128, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Resizing(224, 224, interpolation='bilinear')(inputs)
    x = layers.RandomBrightness(factor=0.1)(x)
    x = layers.RandomContrast(factor=0.1)(x)
    
    base_model = EfficientNetB0(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs, outputs)

tf.keras.utils.set_random_seed(42)
model2_5 = create_efficientnet_resized_model()
model2_5.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Starting Model 2.5 training...")
model2_5.fit(
    X_train_s2_bal, y_train_s2_bal,
    epochs=12,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Explicit save confirmation
save_path = "stage2_model_2_5_efficientnet.keras"
model2_5.save(save_path)

if os.path.exists(save_path):
    print(f"SUCCESS: '{save_path}' was successfully created and verified in {os.getcwd()}")
else:
    print("ERROR: Save failed.")

Checking input arrays...
Balanced training shape: (316, 64, 128, 3)
Starting Model 2.5 training...
Epoch 1/12
8/8 ━━━━━━━━━━━━━━━━━━━━ 39s 3s/step - accuracy: 0.4444 - loss: 0.7219 - val_accuracy: 0.5312 - val_loss: 0.6912
Epoch 2/12
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.4405 - loss: 0.7137 - val_accuracy: 0.5312 - val_loss: 0.6922
Epoch 3/12
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5357 - loss: 0.6887 - val_accuracy: 0.5312 - val_loss: 0.6919
Epoch 4/12
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.4881 - loss: 0.7187 - val_accuracy: 0.5312 - val_loss: 0.6912
Epoch 5/12
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.4802 - loss: 0.7166 - val_accuracy: 0.5312 - val_loss: 0.6912
Epoch 6/12
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5040 - loss: 0.7022 - val_accuracy: 0.5312 - val_loss: 0.6917
Epoch 7/12
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5278 - loss: 0.7027 - val_accuracy: 0.4688 - val_loss: 0.6939
Epoch 8/12
8/8 ━━━━━━━━━━━━

In [17]:
# Cell 9 - Evaluate Resized EfficientNet Model 2.5 & Optimize Threshold

import os
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score, f1_score
import tensorflow as tf

model_path = "/kaggle/input/models/rheincama/model2-5/keras/default/1/stage2_model_2_5_efficientnet.keras"
model2_5 = tf.keras.models.load_model(model_path)

X_test = np.load(os.path.join(WORK_DIR, "X_test_clean.npy"))
y_test = np.load(os.path.join(WORK_DIR, "y_test_clean.npy"))

patho_mask_test = (y_test != 1)
X_test_s2 = X_test[patho_mask_test]
y_test_s2_raw = y_test[patho_mask_test]
y_test_s2_true = np.where(y_test_s2_raw == 0, 0, 1)
probs_s2 = model2_5.predict(X_test_s2).ravel()

# Automated threshold sweep
best_thresh = 0.5
best_macro_f1 = 0.0
best_preds = None

for t in np.arange(0.1, 0.9, 0.01):
    preds = np.where(probs_s2 >= t, 1, 0)
    score = f1_score(y_test_s2_true, preds, average='macro')

    if score > best_macro_f1:
        best_macro_f1 = score
        best_thresh = t
        best_preds = preds
print("=" * 70)
print(f"MODEL 2.5 EFFICIENTNET EVALUATION (Best Threshold = {best_thresh:.2f})")
print("=" * 70)
print(f"Accuracy          : {accuracy_score(y_test_s2_true, best_preds):.4f}")
print(f"Balanced Accuracy : {balanced_accuracy_score(y_test_s2_true, best_preds):.4f}")
print(f"Macro F1          : {f1_score(y_test_s2_true, best_preds, average='macro'):.4f}\n")
print(classification_report(y_test_s2_true, best_preds, target_names=['COPD', 'Pneumonia'], digits=3))
print(pd.DataFrame(

    confusion_matrix(y_test_s2_true, best_preds),
    index=['True COPD', 'True Pneumonia'],
    columns=['Pred COPD', 'Pred Pneumonia']
)) 

31/33 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step

2026-10-10 11:24:53.937095: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:24:54.074100: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:24:54.386682: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:24:54.528408: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:24:55.304638: E external/local_xla/xla/stream_

33/33 ━━━━━━━━━━━━━━━━━━━━ 19s 404ms/step
MODEL 2.5 EFFICIENTNET EVALUATION (Best Threshold = 0.47)
Accuracy          : 0.7248
Balanced Accuracy : 0.6992
Macro F1          : 0.6987

              precision    recall  f1-score   support

        COPD      0.607     0.613     0.610       362
   Pneumonia      0.790     0.785     0.787       670

    accuracy                          0.725      1032
   macro avg      0.698     0.699     0.699      1032
weighted avg      0.726     0.725     0.725      1032

                Pred COPD  Pred Pneumonia
True COPD             222             140
True Pneumonia        144             526


In [18]:
# Complete Sequential Hierarchical Pipeline Evaluation
import os
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from scipy.signal import butter, filtfilt
import xgboost as xgb
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# 1. Load production models
print("Loading Stage 1 (XGBoost)...")
model1_stage1 = xgb.XGBClassifier()
if os.path.exists("stage1_model_1_5.json"):
    model1_stage1.load_model("stage1_model_1_5.json")
else:
    model1_stage1.load_model("stage1_healthy_vs_patho.json")

print("Loading Stage 2 (EfficientNetB0)...")
model2_stage2 = tf.keras.models.load_model("/kaggle/input/models/rheincama/model2-5/keras/default/1/stage2_model_2_5_efficientnet.keras")

# 2. Audio Processing Helper Functions
def bandpass_filter(y, sr, lowcut=100, highcut=2000, order=4):
    nyq = 0.5 * sr
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, y)

def extract_segment_features(y_segment, sr):
    if len(y_segment) < int(sr * 0.2):
        return None
    max_amp = np.max(np.abs(y_segment))
    if max_amp > 0:
        y_segment = y_segment / max_amp
        
    mfcc = librosa.feature.mfcc(y=y_segment, sr=sr, n_mfcc=13, fmin=100, fmax=2000)
    width = 9 if mfcc.shape[1] >= 9 else (mfcc.shape[1] if mfcc.shape[1] % 2 != 0 else mfcc.shape[1] - 1)
    if width >= 3:
        mfcc_delta = librosa.feature.delta(mfcc, width=width)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2, width=width)
    else:
        mfcc_delta = np.zeros_like(mfcc)
        mfcc_delta2 = np.zeros_like(mfcc)
        
    spec_cent = librosa.feature.spectral_centroid(y=y_segment, sr=sr)
    spec_bw = librosa.feature.spectral_bandwidth(y=y_segment, sr=sr)
    spec_contrast = librosa.feature.spectral_contrast(y=y_segment, sr=sr, fmin=100)
    spec_flatness = librosa.feature.spectral_flatness(y=y_segment)
    zcr = librosa.feature.zero_crossing_rate(y=y_segment)
    rms = librosa.feature.rms(y=y_segment)
    
    feats = []
    for mat in [mfcc, mfcc_delta, mfcc_delta2, spec_cent, spec_bw, spec_contrast, spec_flatness, zcr, rms]:
        feats.extend(np.mean(mat, axis=1))
    return np.array(feats, dtype=np.float32)

def extract_cycle_aggregated_features(audio_path, target_sr=16000):
    try:
        y, sr = librosa.load(str(audio_path), sr=target_sr)
        if len(y) == 0:
            return None
        y = bandpass_filter(y, sr)
        
        txt_path = Path(str(audio_path).replace('.wav', '.txt'))
        cycle_bounds = []
        if txt_path.exists():
            with open(txt_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        cycle_bounds.append((float(parts[0]), float(parts[1])))
                        
        if len(cycle_bounds) == 0:
            win_samples = int(2.5 * sr)
            hop_samples = int(1.25 * sr)
            for start in range(0, len(y) - win_samples, hop_samples):
                cycle_bounds.append((start / sr, (start + win_samples) / sr))
                
        cycle_feats = []
        for start_sec, end_sec in cycle_bounds:
            segment = y[int(start_sec * sr):int(end_sec * sr)]
            feat = extract_segment_features(segment, sr)
            if feat is not None:
                cycle_feats.append(feat)
                
        if len(cycle_feats) == 0:
            return None
            
        cycle_mat = np.array(cycle_feats)
        return np.hstack([np.mean(cycle_mat, axis=0), np.max(cycle_mat, axis=0), np.std(cycle_mat, axis=0)])
    except Exception:
        return None

def extract_log_mel_spectrogram(audio_path, target_sr=16000, n_mels=64, max_len=128):
    try:
        y, sr = librosa.load(str(audio_path), sr=target_sr)
        y = bandpass_filter(y, sr)
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, fmin=100, fmax=2000)
        log_mel = librosa.power_to_db(mel, ref=np.max)
        
        if log_mel.shape[1] < max_len:
            pad_width = max_len - log_mel.shape[1]
            log_mel = np.pad(log_mel, ((0, 0), (0, pad_width)), mode='constant')
        else:
            log_mel = log_mel[:, :max_len]
            
        # Stack into 3 channels without altering distribution scale
        log_mel_3ch = np.stack([log_mel]*3, axis=-1)
        return log_mel_3ch
    except Exception:
        return None

# 3. Run Sequential Simulation Loop
test_manifest = pd.read_csv("test_split_manifest.csv")
y_true_seq = []
y_pred_seq = []

print(f"Running sequential pipeline evaluation on {len(test_manifest)} test entries...")

for _, row in test_manifest.iterrows():
    audio_path = row['path']
    true_label = row['label'] # Mapping: 1 = Healthy, 0 = COPD, 2 = Pneumonia
    
    # --- STAGE 1 ---
    feat = extract_cycle_aggregated_features(audio_path)
    if feat is None:
        continue
        
    p_healthy = model1_stage1.predict_proba(feat.reshape(1, -1))[0, 1]
    
    # Threshold 0.61 for Stage 1
    if p_healthy >= 0.61:
        final_pred = 1 # Healthy
    else:
        # --- STAGE 2 ---
        log_mel_3ch = extract_log_mel_spectrogram(audio_path)
        if log_mel_3ch is None:
            continue
            
        p_pneumonia = model2_stage2.predict(log_mel_3ch.reshape(1, 64, 128, 3), verbose=0)[0, 0]
        
        # Threshold 0.47 for Stage 2
        if p_pneumonia >= 0.47:
            final_pred = 2 # Pneumonia
        else:
            final_pred = 0 # COPD
            
    y_true_seq.append(true_label)
    y_pred_seq.append(final_pred)

# 4. Final Metrics Report
y_true_seq = np.array(y_true_seq)
y_pred_seq = np.array(y_pred_seq)

print("=" * 70)
print("SEQUENTIAL HIERARCHICAL PIPELINE EVALUATION REPORT")
print("=" * 70)
print(f"Overall Accuracy : {accuracy_score(y_true_seq, y_pred_seq):.4f}")
print(f"Macro F1-Score   : {f1_score(y_true_seq, y_pred_seq, average='macro', zero_division=0):.4f}\n")

print(classification_report(y_true_seq, y_pred_seq, target_names=['COPD', 'Healthy', 'Pneumonia'], digits=3, zero_division=0))
print(pd.DataFrame(
    confusion_matrix(y_true_seq, y_pred_seq),
    index=['True COPD', 'True Healthy', 'True Pneumonia'],
    columns=['Pred COPD', 'Pred Healthy', 'Pred Pneumonia']
))

Loading Stage 1 (XGBoost)...
Loading Stage 2 (EfficientNetB0)...
Running sequential pipeline evaluation on 263 test entries...


2026-10-10 11:25:08.059220: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:25:08.195265: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:25:08.943273: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-10 11:25:09.079471: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


SEQUENTIAL HIERARCHICAL PIPELINE EVALUATION REPORT
Overall Accuracy : 0.3270
Macro F1-Score   : 0.2978

              precision    recall  f1-score   support

        COPD      0.213     0.860     0.341        50
     Healthy      0.700     0.433     0.535        97
   Pneumonia      1.000     0.009     0.017       116

    accuracy                          0.327       263
   macro avg      0.638     0.434     0.298       263
weighted avg      0.740     0.327     0.270       263

                Pred COPD  Pred Healthy  Pred Pneumonia
True COPD              43             7               0
True Healthy           55            42               0
True Pneumonia        104            11               1
